# Exploring the NWSS Wastewater Series

This notebook shows how to fetch the CDC NWSS series for a state, align it weekly, run both the Farrington and hybrid detectors, and plot the result.

In [ ]:
import asyncio
import matplotlib.pyplot as plt

from argus.ingestion.wastewater.nwss import NWSSIngestor
from argus.fusion.align import to_frame, align_weekly
from argus.evaluation.backtest import run_farrington_backtest
from argus.ingestion.schemas.observation import Metric

obs = asyncio.run(NWSSIngestor().fetch(state="CA"))
df = align_weekly(to_frame(obs))
series = df[df["metric"] == Metric.PATHOGEN_COPIES.value].sort_values("timestamp")
series.head()

In [ ]:
results = run_farrington_backtest(
    df, metric_name=Metric.PATHOGEN_COPIES.value, geo_id="US-CA", warmup=20, z_threshold=1.96
)
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(results["timestamp"], results["observed"], label="observed", color="steelblue")
ax.plot(results["timestamp"], results["threshold"], label="threshold (Farrington)", color="orange", linestyle="--")
alerts = results[results["alert"]]
ax.scatter(alerts["timestamp"], alerts["observed"], color="red", label="alert", zorder=5)
ax.set_ylabel("copies/ml")
ax.set_title("NWSS California — Farrington detection")
ax.legend()
plt.tight_layout()
plt.show()